# Frequentist Hypothesis Testing: Z-Tests and T-Tests

## Overview

In this notebook, we use traditional statistical tests to determine whether email campaigns actually worked.
Instead of running every possible test and hoping for significance, we follow the **Primary → Secondary → Exploratory** framework.
This is the industry standard in A/B testing to control false positive rates and maintain scientific rigor.

## The Framework

### Primary Comparison (Strictest Standard)
- **One** pre-specified business question that we most need answered
- Significance level: **α = 0.05** (5% false positive rate)
- Example: "Did email campaigns increase conversion overall?"

### Secondary Comparisons (Planned Follow-ups)
- **Pre-planned tests** that drill into the primary result
- Significance level: **α = 0.05 / number of tests** (Bonferroni correction)
- Prevents "p-hacking" by adjusting for multiple testing
- Example: "Which email type (Mens vs Womens) performed better?"

### Exploratory Analysis (Hypothesis-Generating)
- Tests we run to **generate ideas** for future experiments
- No statistical correction (but findings require confirmation)
- Clearly labeled as exploratory
- Example: "Does purchase-history alignment matter?"

## Why This Matters

Without discipline, you could run 100 tests and expect ~5 to be "significant" by pure chance.
The Primary → Secondary → Exploratory structure:
- Shows the business we have a real hypothesis
- Controls false positive rate mathematically
- Separates "ideas worth testing" from "findings we're confident in"
- Makes results reproducible and defensible

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t as t_dist, ttest_ind, mannwhitneyu, levene, shapiro
import warnings
warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not available; using matplotlib for all visualizations")

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create output directory
output_dir = '../data/outputs/nb02'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# Load the clean data from nb01
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Check key columns
print(f"\n=== Verification ===")
print(f"'treatment' column exists: {'treatment' in df.columns}")
print(f"'email_match_simple' column exists: {'email_match_simple' in df.columns}")
print(f"\nSegment values in 'segment' column:")
print(df['segment'].value_counts())
print(f"\nTreatment distribution (0=Control, 1=Any Email):")
print(df['treatment'].value_counts())
print(f"\nEmail match distribution:")
print(df['email_match_simple'].value_counts())

## Pre-Specified Analysis Plan

Before we run any tests, we formally commit to this analysis plan.
This prevents "p-hacking" and shows good scientific practice.

### PRIMARY COMPARISON (1 test)
**Question:** Did any email campaign increase conversions overall?

- **H₀ (Null):** Conversion rate (Any Email) = Conversion rate (No Email)
- **H₁ (Alternative):** Conversion rate (Any Email) ≠ Conversion rate (No Email)
- **Significance level:** α = 0.05
- **Test:** Two-proportion Z-test
- **Outcome:** Conversion rate, Visit rate (same question applied to different metrics)

---

### SECONDARY COMPARISONS (4 tests)
**Questions:** Which email type works better, and does it affect spending?

1. **Mens Email vs Control: Conversion rate**
   - H₀: Conv(Mens) = Conv(Control)
   - H₁: Conv(Mens) ≠ Conv(Control)
   
2. **Womens Email vs Control: Conversion rate**
   - H₀: Conv(Womens) = Conv(Control)
   - H₁: Conv(Womens) ≠ Conv(Control)

3. **Mens Email vs Control: Average Spend**
   - H₀: Spend(Mens) = Spend(Control)
   - H₁: Spend(Mens) ≠ Spend(Control)

4. **Womens Email vs Control: Average Spend**
   - H₀: Spend(Womens) = Spend(Control)
   - H₁: Spend(Womens) ≠ Spend(Control)

**Significance level (Bonferroni corrected):** α = 0.05 / 4 = **0.0125 per test**

---

### EXPLORATORY ANALYSIS (6+ tests)
**Questions:** Does the *alignment* between email and purchase history matter?

**Note:** These are hypothesis-generating. We do NOT apply multiple-testing correction, but findings need confirmation in a dedicated follow-up experiment.

- Match (email aligns with history) vs Control: Conversion & Spend
- Mismatch (email doesn't align) vs Control: Conversion & Spend
- Match vs Mismatch direct comparison: Conversion & Spend
- Mixed/Cross-shoppers vs Control: Conversion & Spend

---

## Why Bonferroni Correction?

With 4 secondary tests and α = 0.05 each, the probability of at least one false positive ≈ 18%.
Bonferroni divides the threshold by 4: 0.05 / 4 = 0.0125.
Now the probability of any false positive ≈ 5% (controlling the family-wise error rate).

For exploratory tests, we relax this (no correction) because we're not making business decisions yet—we're just finding hypotheses.

## Hypothesis Testing: A Quick Refresher

### The Basic Idea
We collect data and ask: "Is the difference we see big enough to be real, or could it just be random noise?"

### Key Concepts

**Null Hypothesis (H₀):** There is no difference; any observed difference is random noise.

**Alternative Hypothesis (H₁):** There is a real difference between groups.

**P-value:** The probability of observing data *at least as extreme* as what we saw, **if H₀ were true**.
- Small p-value → unlikely to see this if there were no difference → reject H₀
- Large p-value → plausible under H₀ → fail to reject H₀

**Significance Level (α):** Our threshold for deciding "small enough to reject H₀"
- Typical: α = 0.05 (5% false positive rate)
- More conservative: α = 0.01 (1% false positive rate)

**Type I Error:** Rejecting H₀ when it's actually true (false positive)
- Rate = α

**Type II Error:** Failing to reject H₀ when it's actually false (false negative)
- Related to statistical power

### Tests We'll Use

**Two-Proportion Z-Test:** Comparing conversion rates (binary outcome) between groups
- Test statistic: Z = (p₁ - p₂) / SE
- 95% CI for difference: (p₁ - p₂) ± 1.96 × SE

**Welch's T-Test:** Comparing average spending between groups (continuous outcome)
- Doesn't assume equal variances
- Robust to non-normal data with large samples
- Test statistic: t = (μ₁ - μ₂) / SE

**Mann-Whitney U Test:** Non-parametric alternative to t-test
- Makes no normality assumption
- Good for robustness checks

### Effect Size
Beyond p-values, we report effect size to show *magnitude* of difference:
- **Cohen's h:** For proportions. h = 2×(arcsin(√p₁) - arcsin(√p₂))
  - Small: 0.2, Medium: 0.5, Large: 0.8
- **Cohen's d:** For means. d = (μ₁ - μ₂) / pooled SD
  - Small: 0.2, Medium: 0.5, Large: 0.8

---

# PART 1: PRIMARY COMPARISON

We test the main business question: **Did email campaigns work overall?**

In [ ]:
def two_proportion_ztest(count1, n1, count2, n2):
    """
    Perform a two-proportion Z-test.
    
    Parameters:
    - count1, count2: number of successes in each group
    - n1, n2: sample sizes
    
    Returns:
    - z_stat: Z-statistic
    - p_value: two-tailed p-value
    - ci_diff: 95% CI for (p1 - p2)
    - cohens_h: Cohen's h effect size
    """
    p1 = count1 / n1
    p2 = count2 / n2
    
    # Pooled proportion
    p_pool = (count1 + count2) / (n1 + n2)
    
    # Standard error
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    
    # Z-statistic
    z_stat = (p1 - p2) / se if se > 0 else 0
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    # 95% CI for difference
    se_diff = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    ci_lower = (p1 - p2) - 1.96 * se_diff
    ci_upper = (p1 - p2) + 1.96 * se_diff
    
    # Cohen's h effect size
    h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))
    
    return z_stat, p_value, (ci_lower, ci_upper), p1, p2, h

# PRIMARY: Any Email vs No Email on Conversion
print("="*70)
print("PRIMARY COMPARISON: Any Email vs No Email")
print("Outcome: Conversion Rate")
print("="*70)

# Filter groups
control = df[df['treatment'] == 0]
email = df[df['treatment'] == 1]

print(f"\nSample sizes:")
print(f"  Control (No Email): n = {len(control)}")
print(f"  Email (Any): n = {len(email)}")

# Conversion analysis
control_conv = (control['conversion'] == 1).sum()
email_conv = (email['conversion'] == 1).sum()

control_conv_rate = control_conv / len(control)
email_conv_rate = email_conv / len(email)

z_stat, p_value, ci, p1, p2, h = two_proportion_ztest(
    email_conv, len(email), control_conv, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv}/{len(control)})")
print(f"  Email:   {email_conv_rate:.4f} ({email_conv}/{len(email)})")
print(f"  Difference: {email_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conversion rate (Email) = Conversion rate (Control)")
print(f"  H₁: Conversion rate (Email) ≠ Conversion rate (Control)")
print(f"\n  Z-statistic: {z_stat:.4f}")
print(f"  P-value (two-tailed): {p_value:.6f}")
print(f"  95% CI for difference: [{ci[0]:+.4f}, {ci[1]:+.4f}]")
print(f"  Significance level (α): 0.05")

if p_value < 0.05:
    print(f"\n  ✓ SIGNIFICANT: Reject H₀ (p = {p_value:.6f} < 0.05)")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: Fail to reject H₀ (p = {p_value:.6f} ≥ 0.05)")

# Effect size
if abs(h) < 0.2:
    magnitude = "negligible"
elif abs(h) < 0.5:
    magnitude = "small"
elif abs(h) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"

print(f"\nEffect Size:")
print(f"  Cohen's h: {h:.4f} ({magnitude})")

# Store for later
z_primary_conv = z_stat
p_primary_conv = p_value
h_primary_conv = h

In [ ]:
# PRIMARY: Any Email vs No Email on Visit Rate
print("\n" + "="*70)
print("PRIMARY COMPARISON: Any Email vs No Email")
print("Outcome: Visit Rate")
print("="*70)

# Visit analysis
control_visit = (control['visit'] == 1).sum()
email_visit = (email['visit'] == 1).sum()

control_visit_rate = control_visit / len(control)
email_visit_rate = email_visit / len(email)

z_stat_visit, p_value_visit, ci_visit, pv1, pv2, h_visit = two_proportion_ztest(
    email_visit, len(email), control_visit, len(control)
)

print(f"\nVisit Rates:")
print(f"  Control: {control_visit_rate:.4f} ({control_visit}/{len(control)})")
print(f"  Email:   {email_visit_rate:.4f} ({email_visit}/{len(email)})")
print(f"  Difference: {email_visit_rate - control_visit_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Visit rate (Email) = Visit rate (Control)")
print(f"  H₁: Visit rate (Email) ≠ Visit rate (Control)")
print(f"\n  Z-statistic: {z_stat_visit:.4f}")
print(f"  P-value (two-tailed): {p_value_visit:.6f}")
print(f"  95% CI for difference: [{ci_visit[0]:+.4f}, {ci_visit[1]:+.4f}]")
print(f"  Significance level (α): 0.05")

if p_value_visit < 0.05:
    print(f"\n  ✓ SIGNIFICANT: Reject H₀ (p = {p_value_visit:.6f} < 0.05)")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: Fail to reject H₀ (p = {p_value_visit:.6f} ≥ 0.05)")

if abs(h_visit) < 0.2:
    magnitude = "negligible"
elif abs(h_visit) < 0.5:
    magnitude = "small"
elif abs(h_visit) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"

print(f"\nEffect Size:")
print(f"  Cohen's h: {h_visit:.4f} ({magnitude})")

# Store for later
z_primary_visit = z_stat_visit
p_primary_visit = p_value_visit
h_primary_visit = h_visit

In [ ]:
# PRIMARY: Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['visit', 'conversion', 'spend']
titles = ['Visit Rate', 'Conversion Rate', 'Average Spend ($)']

for ax, metric, title in zip(axes, metrics, titles):
    control_mean = control[metric].mean()
    email_mean = email[metric].mean()
    
    control_se = control[metric].sem()
    email_se = email[metric].sem()
    
    x = [0, 1]
    means = [control_mean, email_mean]
    ses = [control_se, email_se]
    
    colors = ['#1f77b4', '#ff7f0e']
    bars = ax.bar(x, means, yerr=[1.96*se for se in ses], capsize=10, 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(['Control', 'Any Email'])
    ax.set_ylabel(title, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (mean, se) in enumerate(zip(means, ses)):
        ax.text(i, mean + 1.96*se + 0.02*max(means), f'{mean:.3f}', 
                ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('PRIMARY COMPARISON: Any Email vs Control', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_primary_comparison.png', dpi=300, bbox_inches='tight')
print(f"Saved: {output_dir}/nb02_primary_comparison.png")
plt.show()

---

# PART 2: SECONDARY COMPARISONS

We drill into the primary result: which email type (Mens vs Womens) drove the effect,
and does it affect spending?

**Important:** We apply Bonferroni correction across 4 tests (Mens conv, Womens conv, Mens spend, Womens spend).

**Adjusted significance level:** α = 0.05 / 4 = **0.0125**

This means a test must have p < 0.0125 (not just p < 0.05) to be significant at the secondary level.

In [ ]:
print("="*70)
print("SECONDARY COMPARISON: Mens Email vs Control")
print("Outcome: Conversion Rate")
print("="*70)

# Filter segments (exact match: "Mens E-Mail" with no apostrophe)
mens = df[df['segment'] == 'Mens E-Mail']
control = df[df['treatment'] == 0]

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Mens E-Mail: n = {len(mens)}")

# Conversion
mens_conv = (mens['conversion'] == 1).sum()
control_conv = (control['conversion'] == 1).sum()

mens_conv_rate = mens_conv / len(mens)
control_conv_rate = control_conv / len(control)

z_stat_m, p_value_m, ci_m, pm1, pm2, h_m = two_proportion_ztest(
    mens_conv, len(mens), control_conv, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv}/{len(control)})")
print(f"  Mens E-Mail: {mens_conv_rate:.4f} ({mens_conv}/{len(mens)})")
print(f"  Difference: {mens_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conv(Mens) = Conv(Control)")
print(f"  H₁: Conv(Mens) ≠ Conv(Control)")
print(f"\n  Z-statistic: {z_stat_m:.4f}")
print(f"  P-value: {p_value_m:.6f}")
print(f"  95% CI: [{ci_m[0]:+.4f}, {ci_m[1]:+.4f}]")
print(f"  Bonferroni α: 0.0125")

if p_value_m < 0.0125:
    print(f"\n  ✓ SIGNIFICANT: p = {p_value_m:.6f} < 0.0125")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: p = {p_value_m:.6f} ≥ 0.0125")

if abs(h_m) < 0.2:
    mag = "negligible"
elif abs(h_m) < 0.5:
    mag = "small"
elif abs(h_m) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"  Cohen's h: {h_m:.4f} ({mag})")

# Store
sec_mens_conv_z = z_stat_m
sec_mens_conv_p = p_value_m
sec_mens_conv_h = h_m

In [ ]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Womens Email vs Control")
print("Outcome: Conversion Rate")
print("="*70)

# Filter segments (exact match: "Womens E-Mail")
womens = df[df['segment'] == 'Womens E-Mail']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Womens E-Mail: n = {len(womens)}")

# Conversion
womens_conv = (womens['conversion'] == 1).sum()
control_conv_total = (control['conversion'] == 1).sum()

womens_conv_rate = womens_conv / len(womens)
control_conv_rate = control_conv_total / len(control)

z_stat_w, p_value_w, ci_w, pw1, pw2, h_w = two_proportion_ztest(
    womens_conv, len(womens), control_conv_total, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv_total}/{len(control)})")
print(f"  Womens E-Mail: {womens_conv_rate:.4f} ({womens_conv}/{len(womens)})")
print(f"  Difference: {womens_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conv(Womens) = Conv(Control)")
print(f"  H₁: Conv(Womens) ≠ Conv(Control)")
print(f"\n  Z-statistic: {z_stat_w:.4f}")
print(f"  P-value: {p_value_w:.6f}")
print(f"  95% CI: [{ci_w[0]:+.4f}, {ci_w[1]:+.4f}]")
print(f"  Bonferroni α: 0.0125")

if p_value_w < 0.0125:
    print(f"\n  ✓ SIGNIFICANT: p = {p_value_w:.6f} < 0.0125")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: p = {p_value_w:.6f} ≥ 0.0125")

if abs(h_w) < 0.2:
    mag = "negligible"
elif abs(h_w) < 0.5:
    mag = "small"
elif abs(h_w) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"  Cohen's h: {h_w:.4f} ({mag})")

# Store
sec_womens_conv_z = z_stat_w
sec_womens_conv_p = p_value_w
sec_womens_conv_h = h_w

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

segments_list = ['No E-Mail', 'Mens E-Mail', 'Womens E-Mail']
conv_rates = []
conv_ses = []

for seg in segments_list:
    if seg == 'No E-Mail':
        seg_data = control
    else:
        seg_data = df[df['segment'] == seg]
    
    conv_count = (seg_data['conversion'] == 1).sum()
    conv_rate = conv_count / len(seg_data)
    conv_se = np.sqrt(conv_rate * (1 - conv_rate) / len(seg_data))
    
    conv_rates.append(conv_rate)
    conv_ses.append(conv_se)

x = np.arange(len(segments_list))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax.bar(x, conv_rates, yerr=[1.96*se for se in conv_ses], 
               capsize=10, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(segments_list, fontsize=11)
ax.set_ylabel('Conversion Rate', fontsize=12, fontweight='bold')
ax.set_title('SECONDARY COMPARISON: Conversion by Email Type', 
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(conv_rates) * 1.3)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (rate, se) in enumerate(zip(conv_rates, conv_ses)):
    ax.text(i, rate + 1.96*se + 0.01, f'{rate:.3f}', 
            ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_secondary_conversion.png', dpi=300, bbox_inches='tight')
print(f"Saved: {output_dir}/nb02_secondary_conversion.png")
plt.show()

## Spending Analysis: T-Test Assumptions

We now move to spending (a continuous variable), requiring t-tests instead of z-tests.

### When to Use Each Test

**Student's T-Test** (equal variances, rarely correct):
- Assumes both groups have the same variance
- Assumes normal distribution
- Not robust to violations

**Welch's T-Test** (unequal variances):
- Does NOT assume equal variances
- More conservative and safer
- Our default choice

**Mann-Whitney U Test** (non-parametric):
- Makes NO normality assumption
- Uses ranks instead of raw values
- Good robustness check if data is skewed

### Our Strategy

1. Check normality with Shapiro-Wilk test and Q-Q plots
2. Check equal variance with Levene's test
3. Use **Welch's t-test** (the safest bet)
4. Also report **Mann-Whitney U** as a robustness check
5. If both agree, we're confident. If they disagree, be cautious.

In [ ]:
print("="*70)
print("ASSUMPTION CHECKS FOR SPENDING ANALYSIS")
print("="*70)

# Remove zero spending for normality check (common in marketing data)
control_spend_nonzero = control[control['spend'] > 0]['spend'].values
email_spend_nonzero = email[email['spend'] > 0]['spend'].values

# Shapiro-Wilk test (null: data is normal)
stat_c, p_c = shapiro(control_spend_nonzero)
stat_e, p_e = shapiro(email_spend_nonzero)

print(f"\nShapiro-Wilk Normality Test (among non-zero spenders):")
print(f"  Control: W = {stat_c:.4f}, p = {p_c:.6f}")
print(f"    → {'Normal' if p_c > 0.05 else 'NOT Normal'} (p {'>' if p_c > 0.05 else '<'} 0.05)")
print(f"  Email:   W = {stat_e:.4f}, p = {p_e:.6f}")
print(f"    → {'Normal' if p_e > 0.05 else 'NOT Normal'} (p {'>' if p_e > 0.05 else '<'} 0.05)")

# Levene's test for equal variance
stat_lev, p_lev = levene(control['spend'], email['spend'])
print(f"\nLevene's Test for Equal Variance:")
print(f"  Statistic: {stat_lev:.4f}, p = {p_lev:.6f}")
print(f"  → Variances are {'EQUAL' if p_lev > 0.05 else 'UNEQUAL'} (p {'>' if p_lev > 0.05 else '<'} 0.05)")

print(f"\n{'='*70}")
print("CONCLUSION:")
print(f"{'='*70}")
print("Data is NOT normally distributed (common in spending data).")
print("We will use Welch's T-Test (robust to non-normality with large n)")
print("+ Mann-Whitney U (non-parametric robustness check).")

# Q-Q plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

from scipy.stats import probplot

probplot(control_spend_nonzero, dist="norm", plot=axes[0])
axes[0].set_title('Q-Q Plot: Control Group (Non-zero Spenders)', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

probplot(email_spend_nonzero, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: Email Group (Non-zero Spenders)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_qq_plots.png', dpi=300, bbox_inches='tight')
print(f"\nQ-Q plots saved: {output_dir}/nb02_qq_plots.png")
plt.show()

In [ ]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Mens Email vs Control")
print("Outcome: Average Spending")
print("="*70)

# Welch's t-test
t_stat_m_spend, p_value_m_spend = ttest_ind(mens['spend'], control['spend'], 
                                             equal_var=False)

# Mann-Whitney U test
u_stat_m, p_value_m_mw = mannwhitneyu(mens['spend'], control['spend'], 
                                       alternative='two-sided')

# Effect size (Cohen's d)
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

d_m_spend = cohens_d(mens['spend'], control['spend'])

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Mens E-Mail: n = {len(mens)}")

print(f"\nDescriptive Statistics:")
print(f"  Control: Mean = ${control['spend'].mean():.2f}, Median = ${control['spend'].median():.2f}, SD = ${control['spend'].std():.2f}")
print(f"  Mens E-Mail: Mean = ${mens['spend'].mean():.2f}, Median = ${mens['spend'].median():.2f}, SD = ${mens['spend'].std():.2f}")
print(f"  Difference in means: ${mens['spend'].mean() - control['spend'].mean():+.2f}")

print(f"\nWelch's T-Test (Primary, more robust):")
print(f"  H₀: Spend(Mens) = Spend(Control)")
print(f"  H₁: Spend(Mens) ≠ Spend(Control)")
print(f"  t-statistic: {t_stat_m_spend:.4f}")
print(f"  P-value: {p_value_m_spend:.6f}")
print(f"  Bonferroni α: 0.0125")

if p_value_m_spend < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_m_spend:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_m_spend:.6f} ≥ 0.0125")

print(f"\nMann-Whitney U Test (Robustness check, non-parametric):")
print(f"  U-statistic: {u_stat_m:.4f}")
print(f"  P-value: {p_value_m_mw:.6f}")
if p_value_m_mw < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_m_mw:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_m_mw:.6f} ≥ 0.0125")

if abs(d_m_spend) < 0.2:
    mag = "negligible"
elif abs(d_m_spend) < 0.5:
    mag = "small"
elif abs(d_m_spend) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"\nEffect Size:")
print(f"  Cohen's d: {d_m_spend:.4f} ({mag})")

# Store
sec_mens_spend_t = t_stat_m_spend
sec_mens_spend_p = p_value_m_spend
sec_mens_spend_d = d_m_spend

In [ ]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Womens Email vs Control")
print("Outcome: Average Spending")
print("="*70)

# Welch's t-test
t_stat_w_spend, p_value_w_spend = ttest_ind(womens['spend'], control['spend'], 
                                             equal_var=False)

# Mann-Whitney U test
u_stat_w, p_value_w_mw = mannwhitneyu(womens['spend'], control['spend'], 
                                       alternative='two-sided')

# Effect size
d_w_spend = cohens_d(womens['spend'], control['spend'])

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Womens E-Mail: n = {len(womens)}")

print(f"\nDescriptive Statistics:")
print(f"  Control: Mean = ${control['spend'].mean():.2f}, Median = ${control['spend'].median():.2f}, SD = ${control['spend'].std():.2f}")
print(f"  Womens E-Mail: Mean = ${womens['spend'].mean():.2f}, Median = ${womens['spend'].median():.2f}, SD = ${womens['spend'].std():.2f}")
print(f"  Difference in means: ${womens['spend'].mean() - control['spend'].mean():+.2f}")

print(f"\nWelch's T-Test (Primary, more robust):")
print(f"  H₀: Spend(Womens) = Spend(Control)")
print(f"  H₁: Spend(Womens) ≠ Spend(Control)")
print(f"  t-statistic: {t_stat_w_spend:.4f}")
print(f"  P-value: {p_value_w_spend:.6f}")
print(f"  Bonferroni α: 0.0125")

if p_value_w_spend < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_w_spend:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_w_spend:.6f} ≥ 0.0125")

print(f"\nMann-Whitney U Test (Robustness check, non-parametric):")
print(f"  U-statistic: {u_stat_w:.4f}")
print(f"  P-value: {p_value_w_mw:.6f}")
if p_value_w_mw < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_w_mw:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_w_mw:.6f} ≥ 0.0125")

if abs(d_w_spend) < 0.2:
    mag = "negligible"
elif abs(d_w_spend) < 0.5:
    mag = "small"
elif abs(d_w_spend) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"\nEffect Size:")
print(f"  Cohen's d: {d_w_spend:.4f} ({mag})")

# Store
sec_womens_spend_t = t_stat_w_spend
sec_womens_spend_p = p_value_w_spend
sec_womens_spend_d = d_w_spend

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

segments_list = ['No E-Mail', 'Mens E-Mail', 'Womens E-Mail']
spend_data = []
spend_means = []
spend_sds = []

for seg in segments_list:
    if seg == 'No E-Mail':
        seg_data = control['spend'].values
    else:
        seg_data = df[df['segment'] == seg]['spend'].values
    
    spend_data.append(seg_data)
    spend_means.append(np.mean(seg_data))
    spend_sds.append(np.std(seg_data))

# Create box plot (without outliers for clarity)
bp = ax.boxplot(spend_data, labels=segments_list, patch_artist=True,
                 showfliers=False, widths=0.6)

# Color boxes
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay means
x_pos = np.arange(1, len(segments_list) + 1)
ax.scatter(x_pos, spend_means, color='red', s=150, marker='D', 
          zorder=3, label='Mean', edgecolors='darkred', linewidth=2)

ax.set_ylabel('Spending ($)', fontsize=12, fontweight='bold')
ax.set_title('SECONDARY COMPARISON: Spending by Email Type', 
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=10)

# Add mean values as text
for i, mean in enumerate(spend_means):
    ax.text(i+1, mean + 20, f'${mean:.2f}', ha='center', va='bottom', 
            fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_secondary_spending.png', dpi=300, bbox_inches='tight')
print(f"Saved: {output_dir}/nb02_secondary_spending.png")
plt.show()

---

# PART 3: EXPLORATORY ANALYSIS

## Purchase History Match: Do We Send the Right Emails?

**Important Disclaimer:** These analyses are **hypothesis-generating only**.
We do NOT control the false positive rate (no Bonferroni correction).
Any findings must be confirmed in a dedicated follow-up experiment.

### What is Email Match?

The data includes a feature `email_match_simple` based on **purchase department history**, NOT customer gender:

- **Match:** Customer's email aligns with their shopping history
  - Example: A "womens_only" buyer receives "Womens E-Mail"
  
- **Mismatch:** Customer's email doesn't align with their history
  - Example: A "mens_only" buyer receives "Womens E-Mail"

- **Mixed:** Customer is a cross-shopper (buys both departments)

- **Control:** No email sent

### Research Question
Does alignment between email and purchase history improve performance?
This tests whether we're "preaching to the choir" or genuinely reaching the right audience.

In [ ]:
print("\n" + "="*70)
print("EXPLORATORY: Match vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)

# Filter Match and Control
match = df[df['email_match_simple'] == 'Match']
control_exp = df[df['email_match_simple'] == 'Control']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Match: n = {len(match)}")

# === Conversion ===
match_conv = (match['conversion'] == 1).sum()
control_conv = (control_exp['conversion'] == 1).sum()

match_conv_rate = match_conv / len(match)
control_conv_rate = control_conv / len(control_exp)

z_stat_match_conv, p_value_match_conv, ci_match_conv, _, _, h_match_conv = two_proportion_ztest(
    match_conv, len(match), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Match: {match_conv_rate:.4f} ({match_conv}/{len(match)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {match_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_match_conv:.4f}, p = {p_value_match_conv:.6f}")
if p_value_match_conv < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_match_conv:.4f}")

# === Spending ===
t_stat_match_spend, p_value_match_spend = ttest_ind(match['spend'], control_exp['spend'], 
                                                      equal_var=False)
d_match_spend = cohens_d(match['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Match Mean: ${match['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${match['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_match_spend:.4f}, p = {p_value_match_spend:.6f}")
if p_value_match_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_match_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")

In [ ]:
print("\n" + "="*70)
print("EXPLORATORY: Mismatch vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)

# Filter Mismatch
mismatch = df[df['email_match_simple'] == 'Mismatch']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Mismatch: n = {len(mismatch)}")

# === Conversion ===
mismatch_conv = (mismatch['conversion'] == 1).sum()
mismatch_conv_rate = mismatch_conv / len(mismatch)

z_stat_mmatch_conv, p_value_mmatch_conv, ci_mmatch_conv, _, _, h_mmatch_conv = two_proportion_ztest(
    mismatch_conv, len(mismatch), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Mismatch: {mismatch_conv_rate:.4f} ({mismatch_conv}/{len(mismatch)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {mismatch_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mmatch_conv:.4f}, p = {p_value_mmatch_conv:.6f}")
if p_value_mmatch_conv < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mmatch_conv:.4f}")

# === Spending ===
t_stat_mmatch_spend, p_value_mmatch_spend = ttest_ind(mismatch['spend'], control_exp['spend'], 
                                                        equal_var=False)
d_mmatch_spend = cohens_d(mismatch['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Mismatch Mean: ${mismatch['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${mismatch['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mmatch_spend:.4f}, p = {p_value_mmatch_spend:.6f}")
if p_value_mmatch_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mmatch_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")

In [ ]:
print("\n" + "="*70)
print("EXPLORATORY: Match vs Mismatch (Direct Comparison)")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)
print("Question: Does email alignment actually improve performance?")

print(f"\nSample sizes:")
print(f"  Match: n = {len(match)}")
print(f"  Mismatch: n = {len(mismatch)}")

# === Conversion ===
match_conv_rate = (match['conversion'] == 1).sum() / len(match)
mismatch_conv_rate = (mismatch['conversion'] == 1).sum() / len(mismatch)

z_stat_mvsmm, p_value_mvsmm, ci_mvsmm, _, _, h_mvsmm = two_proportion_ztest(
    (match['conversion'] == 1).sum(), len(match), 
    (mismatch['conversion'] == 1).sum(), len(mismatch)
)

print(f"\n--- CONVERSION ---")
print(f"Match: {match_conv_rate:.4f}")
print(f"Mismatch: {mismatch_conv_rate:.4f}")
print(f"Difference: {match_conv_rate - mismatch_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mvsmm:.4f}, p = {p_value_mvsmm:.6f}")
if p_value_mvsmm < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mvsmm:.4f}")

# === Spending ===
t_stat_mvsmm_spend, p_value_mvsmm_spend = ttest_ind(match['spend'], mismatch['spend'], 
                                                      equal_var=False)
d_mvsmm_spend = cohens_d(match['spend'], mismatch['spend'])

print(f"\n--- SPENDING ---")
print(f"Match Mean: ${match['spend'].mean():.2f}")
print(f"Mismatch Mean: ${mismatch['spend'].mean():.2f}")
print(f"Difference: ${match['spend'].mean() - mismatch['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mvsmm_spend:.4f}, p = {p_value_mvsmm_spend:.6f}")
if p_value_mvsmm_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mvsmm_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")

In [ ]:
print("\n" + "="*70)
print("EXPLORATORY: Mixed (Cross-shoppers) vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)
print("Question: How do cross-shoppers respond to email campaigns?")

# Filter Mixed
mixed = df[df['email_match_simple'] == 'Mixed']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Mixed: n = {len(mixed)}")

# === Conversion ===
mixed_conv = (mixed['conversion'] == 1).sum()
mixed_conv_rate = mixed_conv / len(mixed)

z_stat_mixed, p_value_mixed, ci_mixed, _, _, h_mixed = two_proportion_ztest(
    mixed_conv, len(mixed), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Mixed: {mixed_conv_rate:.4f} ({mixed_conv}/{len(mixed)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {mixed_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mixed:.4f}, p = {p_value_mixed:.6f}")
if p_value_mixed < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mixed:.4f}")

# === Spending ===
t_stat_mixed_spend, p_value_mixed_spend = ttest_ind(mixed['spend'], control_exp['spend'], 
                                                      equal_var=False)
d_mixed_spend = cohens_d(mixed['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Mixed Mean: ${mixed['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${mixed['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mixed_spend:.4f}, p = {p_value_mixed_spend:.6f}")
if p_value_mixed_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mixed_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conversion rates by match type
segments_exp = ['Control', 'Match', 'Mismatch', 'Mixed']
groups_exp = [control_exp, match, mismatch, mixed]
conv_rates_exp = [(g['conversion'] == 1).sum() / len(g) for g in groups_exp]
conv_ses_exp = [np.sqrt(cr * (1 - cr) / len(g)) for cr, g in zip(conv_rates_exp, groups_exp)]

colors_exp = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
x_exp = np.arange(len(segments_exp))

bars1 = axes[0].bar(x_exp, conv_rates_exp, yerr=[1.96*se for se in conv_ses_exp], 
                    capsize=10, color=colors_exp, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_xticks(x_exp)
axes[0].set_xticklabels(segments_exp, fontsize=10)
axes[0].set_ylabel('Conversion Rate', fontsize=11, fontweight='bold')
axes[0].set_title('EXPLORATORY: Conversion by Match Status', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, (rate, se) in enumerate(zip(conv_rates_exp, conv_ses_exp)):
    axes[0].text(i, rate + 1.96*se + 0.005, f'{rate:.3f}', 
                ha='center', va='bottom', fontweight='bold', fontsize=9)

# Spending by match type
spend_means_exp = [g['spend'].mean() for g in groups_exp]

bars2 = axes[1].bar(x_exp, spend_means_exp, color=colors_exp, alpha=0.7, 
                    edgecolor='black', linewidth=1.5)
axes[1].set_xticks(x_exp)
axes[1].set_xticklabels(segments_exp, fontsize=10)
axes[1].set_ylabel('Average Spending ($)', fontsize=11, fontweight='bold')
axes[1].set_title('EXPLORATORY: Spending by Match Status', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, mean in enumerate(spend_means_exp):
    axes[1].text(i, mean + 10, f'${mean:.2f}', 
                ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.suptitle('EXPLORATORY ANALYSIS: Email Alignment Effects', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_exploratory_match.png', dpi=300, bbox_inches='tight')
print(f"Saved: {output_dir}/nb02_exploratory_match.png")
plt.show()

## Effect Size Interpretation

Beyond p-values, effect size tells us the **magnitude** of the difference.
A tiny effect might be statistically significant (p < 0.05) in a large sample,
but practically meaningless.

### Cohen's h (for proportions/conversions)
Measures difference between two proportions.

- **h < 0.2:** Negligible effect
- **0.2 ≤ h < 0.5:** Small effect
- **0.5 ≤ h < 0.8:** Medium effect
- **h ≥ 0.8:** Large effect

### Cohen's d (for means/spending)
Measures difference between two group means, standardized by pooled SD.

- **d < 0.2:** Negligible effect
- **0.2 ≤ d < 0.5:** Small effect
- **0.5 ≤ d < 0.8:** Medium effect
- **d ≥ 0.8:** Large effect

### Combined Interpretation
For strong evidence, we want:
1. **Statistically significant** (p-value below threshold)
2. **Practically meaningful** effect size (d or h > 0.2)
3. **Consistent direction** (same sign across related tests)
4. **Robust** (confirmed by multiple tests, e.g., t-test + Mann-Whitney)

In [ ]:
# Build comprehensive results table
results = []

# PRIMARY
results.append({
    'Level': 'PRIMARY',
    'Outcome': 'Conversion',
    'Comparison': 'Any Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{email_conv_rate:.4f}',
    'Difference': f'{email_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_primary_conv:.4f}',
    'P_Value': f'{p_primary_conv:.6f}',
    'Alpha_Threshold': '0.0500',
    'Significant': 'Yes' if p_primary_conv < 0.05 else 'No',
    'Effect_Size': f'{h_primary_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_primary_conv) < 0.5 else ('Medium' if abs(h_primary_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'PRIMARY',
    'Outcome': 'Visit',
    'Comparison': 'Any Email vs Control',
    'Control_Rate': f'{control_visit_rate:.4f}',
    'Treatment_Rate': f'{email_visit_rate:.4f}',
    'Difference': f'{email_visit_rate - control_visit_rate:+.4f}',
    'Test_Statistic': f'{z_primary_visit:.4f}',
    'P_Value': f'{p_primary_visit:.6f}',
    'Alpha_Threshold': '0.0500',
    'Significant': 'Yes' if p_primary_visit < 0.05 else 'No',
    'Effect_Size': f'{h_primary_visit:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_primary_visit) < 0.5 else ('Medium' if abs(h_primary_visit) < 0.8 else 'Large')
})

# SECONDARY
results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Conversion',
    'Comparison': 'Mens Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mens_conv_rate:.4f}',
    'Difference': f'{mens_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{sec_mens_conv_z:.4f}',
    'P_Value': f'{sec_mens_conv_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_mens_conv_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_mens_conv_h:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_mens_conv_h) < 0.5 else ('Medium' if abs(sec_mens_conv_h) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Conversion',
    'Comparison': 'Womens Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{womens_conv_rate:.4f}',
    'Difference': f'{womens_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{sec_womens_conv_z:.4f}',
    'P_Value': f'{sec_womens_conv_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_womens_conv_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_womens_conv_h:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_womens_conv_h) < 0.5 else ('Medium' if abs(sec_womens_conv_h) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Spending',
    'Comparison': 'Mens Email vs Control',
    'Control_Rate': f'${control["spend"].mean():.2f}',
    'Treatment_Rate': f'${mens["spend"].mean():.2f}',
    'Difference': f'${mens["spend"].mean() - control["spend"].mean():+.2f}',
    'Test_Statistic': f'{sec_mens_spend_t:.4f}',
    'P_Value': f'{sec_mens_spend_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_mens_spend_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_mens_spend_d:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_mens_spend_d) < 0.5 else ('Medium' if abs(sec_mens_spend_d) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Spending',
    'Comparison': 'Womens Email vs Control',
    'Control_Rate': f'${control["spend"].mean():.2f}',
    'Treatment_Rate': f'${womens["spend"].mean():.2f}',
    'Difference': f'${womens["spend"].mean() - control["spend"].mean():+.2f}',
    'Test_Statistic': f'{sec_womens_spend_t:.4f}',
    'P_Value': f'{sec_womens_spend_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_womens_spend_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_womens_spend_d:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_womens_spend_d) < 0.5 else ('Medium' if abs(sec_womens_spend_d) < 0.8 else 'Large')
})

# EXPLORATORY
results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Match vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{match_conv_rate:.4f}',
    'Difference': f'{match_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_match_conv:.4f}',
    'P_Value': f'{p_value_match_conv:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_match_conv < 0.05 else 'No',
    'Effect_Size': f'{h_match_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_match_conv) < 0.5 else ('Medium' if abs(h_match_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Match vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${match["spend"].mean():.2f}',
    'Difference': f'${match["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_match_spend:.4f}',
    'P_Value': f'{p_value_match_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_match_spend < 0.05 else 'No',
    'Effect_Size': f'{d_match_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_match_spend) < 0.5 else ('Medium' if abs(d_match_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Mismatch vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mismatch_conv_rate:.4f}',
    'Difference': f'{mismatch_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mmatch_conv:.4f}',
    'P_Value': f'{p_value_mmatch_conv:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mmatch_conv < 0.05 else 'No',
    'Effect_Size': f'{h_mmatch_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mmatch_conv) < 0.5 else ('Medium' if abs(h_mmatch_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Mismatch vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${mismatch["spend"].mean():.2f}',
    'Difference': f'${mismatch["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mmatch_spend:.4f}',
    'P_Value': f'{p_value_mmatch_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mmatch_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mmatch_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mmatch_spend) < 0.5 else ('Medium' if abs(d_mmatch_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Match vs Mismatch',
    'Control_Rate': f'{mismatch_conv_rate:.4f}',
    'Treatment_Rate': f'{match_conv_rate:.4f}',
    'Difference': f'{match_conv_rate - mismatch_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mvsmm:.4f}',
    'P_Value': f'{p_value_mvsmm:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mvsmm < 0.05 else 'No',
    'Effect_Size': f'{h_mvsmm:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mvsmm) < 0.5 else ('Medium' if abs(h_mvsmm) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Match vs Mismatch',
    'Control_Rate': f'${mismatch["spend"].mean():.2f}',
    'Treatment_Rate': f'${match["spend"].mean():.2f}',
    'Difference': f'${match["spend"].mean() - mismatch["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mvsmm_spend:.4f}',
    'P_Value': f'{p_value_mvsmm_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mvsmm_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mvsmm_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mvsmm_spend) < 0.5 else ('Medium' if abs(d_mvsmm_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Mixed vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mixed_conv_rate:.4f}',
    'Difference': f'{mixed_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mixed:.4f}',
    'P_Value': f'{p_value_mixed:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mixed < 0.05 else 'No',
    'Effect_Size': f'{h_mixed:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mixed) < 0.5 else ('Medium' if abs(h_mixed) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Mixed vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${mixed["spend"].mean():.2f}',
    'Difference': f'${mixed["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mixed_spend:.4f}',
    'P_Value': f'{p_value_mixed_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mixed_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mixed_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mixed_spend) < 0.5 else ('Medium' if abs(d_mixed_spend) < 0.8 else 'Large')
})

results_df = pd.DataFrame(results)
results_df.to_csv(f'{output_dir}/nb02_frequentist_results.csv', index=False)

print("\n" + "="*100)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*100)
print(results_df.to_string(index=False))
print("\n" + "="*100)
print(f"Saved to: {output_dir}/nb02_frequentist_results.csv"
      f"\n

In [ ]:
# Create a summary visualization (forest plot style)
fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# === PRIMARY RESULTS ===
primary_tests = ['Conv (Any Email)', 'Visit (Any Email)']
primary_ps = [p_primary_conv, p_primary_visit]
primary_colors = ['green' if p < 0.05 else 'red' for p in primary_ps]

y_pos = np.arange(len(primary_tests))
bars1 = axes[0].barh(y_pos, primary_ps, color=primary_colors, alpha=0.6, edgecolor='black', linewidth=1.5)
axes[0].axvline(x=0.05, color='black', linestyle='--', linewidth=2, label='α = 0.05')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(primary_tests, fontsize=11)
axes[0].set_xlabel('P-value', fontsize=11, fontweight='bold')
axes[0].set_title('PRIMARY RESULTS (α = 0.05)', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].grid(axis='x', alpha=0.3)
axes[0].legend(fontsize=10)

for i, (p, test) in enumerate(zip(primary_ps, primary_tests)):
    axes[0].text(p + 0.02, i, f'p={p:.4f}', va='center', fontweight='bold', fontsize=10)

# === SECONDARY RESULTS ===
secondary_tests = [
    'Conv: Mens',
    'Conv: Womens',
    'Spend: Mens',
    'Spend: Womens'
]
secondary_ps = [sec_mens_conv_p, sec_womens_conv_p, sec_mens_spend_p, sec_womens_spend_p]
secondary_colors = ['green' if p < 0.0125 else 'red' for p in secondary_ps]

y_pos_sec = np.arange(len(secondary_tests))
bars2 = axes[1].barh(y_pos_sec, secondary_ps, color=secondary_colors, alpha=0.6, 
                     edgecolor='black', linewidth=1.5)
axes[1].axvline(x=0.0125, color='black', linestyle='--', linewidth=2, label='α = 0.0125 (Bonferroni)')
axes[1].set_yticks(y_pos_sec)
axes[1].set_yticklabels(secondary_tests, fontsize=11)
axes[1].set_xlabel('P-value', fontsize=11, fontweight='bold')
axes[1].set_title('SECONDARY RESULTS (α = 0.05/4 = 0.0125, Bonferroni Corrected)', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 1)
axes[1].grid(axis='x', alpha=0.3)
axes[1].legend(fontsize=10)

for i, (p, test) in enumerate(zip(secondary_ps, secondary_tests)):
    axes[1].text(p + 0.02, i, f'p={p:.4f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{output_dir}/nb02_results_summary.png', dpi=300, bbox_inches='tight')
print(f"Saved: {output_dir}/nb02_results_summary.png")
plt.show()

---

# Key Takeaways

## PRIMARY LEVEL: Did Email Campaigns Work?

Review the p-values and effect sizes for the primary tests (Any Email vs Control).
- If **p < 0.05 AND effect size is non-negligible**, email campaigns had a real impact overall.
- If **p ≥ 0.05**, we cannot claim the campaigns worked (lack of evidence).
- Effect size matters: even a significant p-value is weak if Cohen's h or d < 0.2.

## SECONDARY LEVEL: Which Email Type Performed Better?

Look at Mens Email vs Control and Womens Email vs Control (both conversion and spending).
- Remember: these tests use **α = 0.0125** (stricter than primary).
- Only results with **p < 0.0125** are statistically significant at the secondary level.
- Bonferroni correction prevents false claims about which email variant "works best."

## EXPLORATORY LEVEL: Does Email Alignment Matter?

The Match vs Mismatch analyses explore whether targeting based on purchase history helps.
- **These are ideas for future experiments, not confirmed findings.**
- No correction applied (no Bonferroni), so p < 0.05 is suggestive but not conclusive.
- Any interesting patterns should be tested in a dedicated follow-up experiment.

---

## What to Report to Stakeholders

**Frame the analysis clearly:**
- "We had one primary question, four pre-planned secondary comparisons, and exploratory sub-analyses."
- "Our primary comparison tests whether ANY email helped. Secondary tests drill into WHICH email type."
- "The exploratory analyses generate hypotheses (email alignment, cross-shopper behavior) for future testing."

**If primary result is significant:**
- "Email campaigns improved [conversion/visits/spending] (p < 0.05, effect size = X)."
- "This effect appears driven by [Mens/Womens] email (if secondary is also significant)."
- "However, [other observations from exploratory] suggest additional opportunities worth testing."

**If primary result is NOT significant:**
- "Email campaigns did not produce a detectable effect overall (p ≥ 0.05)."
- "Secondary analyses show [details], but these are exploratory given the null primary result."
- "Implications: Either the email strategy needs redesign, or effect is too small to detect with current sample size."

---

## Next Steps

1. **Validate primary finding:** If significant, replicate in a new holdout sample or test period.
2. **Test exploratory hypotheses:** Design a dedicated experiment around email alignment or other promising patterns.
3. **Dig into mechanisms:** Why did email (or not) work? Analyze open rates, click rates, timing, content.
4. **Power analysis:** If effect size is small, consider whether larger sample is needed to detect it reliably.
5. **Subgroup analysis:** Conduct planned (not post-hoc) analysis by customer segment, RFM tier, etc.

---

## Common Pitfalls to Avoid

- ✗ **P-hacking:** Running 50 tests and reporting only the significant ones. (We avoided this with our pre-specified plan.)
- ✗ **Ignoring effect size:** A p-value of 0.04 with h = 0.05 is not practically meaningful.
- ✗ **Overgeneralizing exploratory findings:** Just because Match vs Control is significant doesn't mean we should send only matched emails—test it first!
- ✗ **Forgetting multiple testing correction:** Secondary tests need stricter thresholds.
- ✗ **Confusing absence of evidence with evidence of absence:** p ≥ 0.05 means "no strong evidence," not "proved null hypothesis."

---

## Statistical Recap

| Level | Tests | Significance Level | Correction | Purpose |
|-------|-------|-------------------|-----------|---------|
| **Primary** | 1-2 | α = 0.05 | None | Main business question |
| **Secondary** | 4 | α = 0.0125 | Bonferroni | Pre-planned follow-ups |
| **Exploratory** | 6+ | None (p < 0.05 suggestive) | None | Hypothesis generation |

Use this framework to maintain scientific rigor while staying responsive to business needs.